# 02 - Preprocessing: from waveforms to spectrograms and scalograms

For every record:
1. the mean of the recorded samples is removed from each component (zero padding stays zero);
2. the analysis window is placed around the peak absolute acceleration over the three components (PGA),
   from `pre_pga_s` before to `post_pga_s` after; the same window is used for the three components;
3. the window, extended by a context margin on both sides, is tapered, low-pass filtered and resampled;
4. spectrogram (STFT) and scalogram (Morlet CWT) are computed on the extended window and cropped to the window,
   on a common grid of logarithmic frequencies and time bins;
5. the power is converted to absolute decibels and stored as arrays of shape (components, frequencies, times).

Every parameter is read from the `preprocessing` section of `configs/default.yaml`.
The records come from `waveforms.source` (development subset now, full dataset later).

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wqu.config import load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.inspection import share_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.sampling import sample_records_per_label
from wqu.data.waveforms import (
    iter_waveform_batches,
    load_waveforms,
    select_inspection_records,
    stored_sampling_rate_hz,
)
from wqu.preprocessing.pipeline import (
    FEATURE_INDEX_COLUMN,
    REPRESENTATIONS,
    feature_file_path,
    grid_from_config,
    pooled_power_values,
    preprocess_and_store,
    process_record,
    read_features,
    window_table,
)
from wqu.preprocessing.transform import power_to_db
from wqu.preprocessing.window_search import (
    collect_window_metrics,
    pareto_front,
    select_window,
    summarize_window_grid,
    window_row,
)
from wqu.visualization import (
    apply_plot_style,
    plot_representation_comparison,
    plot_value_distributions,
    plot_window_grid_heatmap,
    plot_window_overlay,
    plot_window_tradeoff,
    save_figure,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
apply_plot_style()
columns = config.columns
preprocessing = config.preprocessing
tolerance = config.waveforms.zero_tolerance
label_names = {subset.label: subset.name for subset in config.data.subsets}
for directory in (preprocessing.figures_dir, preprocessing.tables_dir, preprocessing.figures_dir / "png_checks"):
    directory.mkdir(parents=True, exist_ok=True)

grid = grid_from_config(preprocessing)
print(f"Window: {preprocessing.pre_pga_s:g} s before to {preprocessing.post_pga_s:g} s after the PGA "
      f"(+ {preprocessing.context_margin_s:g} s margin on each side)")
print(f"Resampling: {preprocessing.target_sampling_rate_hz:g} Hz after a low-pass at {preprocessing.antialias_corner_hz:g} Hz")
print(f"Grid: {preprocessing.n_frequencies} frequencies from {preprocessing.min_frequency_hz:g} to "
      f"{preprocessing.max_frequency_hz:g} Hz (log), {preprocessing.n_times} time bins of "
      f"{preprocessing.window_length_s / preprocessing.n_times:.3f} s")
print(f"STFT window: {preprocessing.stft_window_s:g} s ({preprocessing.stft_window_type}); "
      f"Morlet w0 = {preprocessing.morlet_w0:g}")

In [ ]:
_, metadata = load_esm25_metadata(config)
records = select_inspection_records(metadata, config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
print(f"Waveform source: {config.waveforms.source}; records: {len(records):,}")
records[columns.label].value_counts().rename(index=label_names)

## 1. Analysis window: what is kept and what is cut

For every record:
- `pga_time_s`: time of the PGA from the start of the stored trace;
- `nonzero_cut_before_c`, `nonzero_cut_after_c`: recorded (non-zero) samples of component c removed before and after the window;
- `missing_in_window_s_c`: seconds of the window without data on component c (window reaching beyond the recording);
- `recorded_fraction_kept`, `energy_fraction_kept`: share of the recorded samples and of the signal energy (sum of squared amplitudes) inside the window.

In [ ]:
windows = window_table(iter_waveform_batches(records, config), sampling_rate_hz, preprocessing, tolerance)
windows = records[[columns.label, "trace_name_old", columns.station_id]].join(windows)
for side in ("before", "after"):
    windows[f"cut_{side}_s"] = windows[[f"nonzero_cut_{side}_{c}" for c in CV_COMPONENTS]].max(axis=1) / sampling_rate_hz
windows["missing_in_window_s"] = windows[[f"missing_in_window_s_{c}" for c in CV_COMPONENTS]].max(axis=1)
windows.to_csv(preprocessing.tables_dir / f"window_statistics_{config.waveforms.source}.csv")

WINDOW_COLUMNS = ["pga_time_s", "cut_before_s", "cut_after_s", "missing_in_window_s", "recorded_fraction_kept", "energy_fraction_kept"]
numeric_summary_by_label(windows, WINDOW_COLUMNS, columns.label).rename(index=label_names, level=columns.label).round(3)

In [ ]:
window_flags = pd.DataFrame(
    {
        "window starts before the recording": windows["window_start"] < 0,
        "window ends after the stored trace": windows["window_stop"] > config.data.waveform_length_s * sampling_rate_hz,
        "some data missing in the window": windows["missing_in_window_s"] > 0,
        "energy kept < 95%": windows["energy_fraction_kept"] < 0.95,
        "energy kept < 80%": windows["energy_fraction_kept"] < 0.80,
    }
)
share_by_label(window_flags, windows[columns.label]).rename(columns=label_names).round(3)

## 2. Padding leakage and choice of the window

The zero padding of the stored traces differs between the classes (notebook 01: the trailing padding is longer
in good records). If a window reaches into the padding, the amount of padding it contains can reveal the class
without looking at the waveform. For every candidate window from `window_search.pre_pga_s` before to
`window_search.post_pga_s` after the PGA (plus the configured window) two properties are measured:

- **leakage**: seconds of padding in the window (before the PGA, after the PGA, total; worst component);
  for each of the three, the class separation max(AUC, 1 - AUC), where 0.5 means no information.
  A window is accepted when all three are at most `window_search.max_padding_separation`;
- **signal loss**: fraction of the energy of the stored trace kept in the window, on the records of
  `window_search.energy.reference_label`, summarized by `window_search.energy.criterion`.

Among accepted windows the one with the highest energy score is suggested, choosing the shortest among scores
within `window_search.score_tolerance`. The suggestion is only printed: the preprocessing always uses
`window.pre_pga_s` and `window.post_pga_s` of the configuration, which must be updated by hand.

The separation is measured on the records of this notebook (ESM25, used for training and validation only),
so the choice of the window does not use test data.

In [ ]:
search = preprocessing.window_search
window_metrics = collect_window_metrics(
    iter_waveform_batches(records, config),
    sampling_rate_hz,
    pre_pga_s=(*search.pre_pga_s, preprocessing.pre_pga_s),
    post_pga_s=(*search.post_pga_s, preprocessing.post_pga_s),
    tolerance=tolerance,
    label_column=columns.label,
)
window_summary = summarize_window_grid(window_metrics, search)
window_summary["pareto"] = pareto_front(window_summary)
window_summary.to_csv(preprocessing.tables_dir / f"window_search_{config.waveforms.source}.csv", index=False)

configured = window_row(window_summary, preprocessing.pre_pga_s, preprocessing.post_pga_s)
print(f"Configured window ({preprocessing.pre_pga_s:g} s, {preprocessing.post_pga_s:g} s): "
      f"padding separation {configured['padding_separation']:.3f} "
      f"(threshold {search.max_padding_separation:g}), energy score {configured['energy_score']:.3f}, "
      f"{'accepted' if configured['accepted'] else 'rejected'}")
configured.to_frame("configured window")

In [ ]:
suggested = select_window(window_summary, search)
print(f"Suggested window: pre_pga_s = {suggested['pre_pga_s']:g}, post_pga_s = {suggested['post_pga_s']:g} "
      f"(length {suggested['window_length_s']:g} s), padding separation {suggested['padding_separation']:.3f}, "
      f"energy score {suggested['energy_score']:.3f}")
print("To use it, set preprocessing.window.pre_pga_s and post_pga_s in the configuration and rerun the notebook.")

SUMMARY_COLUMNS = ["pre_pga_s", "post_pga_s", "window_length_s", "separation_padding_before_pga_s",
                   "separation_padding_after_pga_s", "separation_padding_total_s", "padding_separation",
                   "share_with_padding_0", "share_with_padding_1", "energy_score", "median_energy_fraction_kept",
                   "accepted"]
window_summary[window_summary["pareto"]].sort_values("padding_separation")[SUMMARY_COLUMNS].round(3)

In [ ]:
marked_windows = {
    "suggested": (suggested["pre_pga_s"], suggested["post_pga_s"]),
    "configured": (preprocessing.pre_pga_s, preprocessing.post_pga_s),
}
grid_summary = window_summary[window_summary["pre_pga_s"].isin(search.pre_pga_s)
                              & window_summary["post_pga_s"].isin(search.post_pga_s)]
figure, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
plot_window_grid_heatmap(grid_summary, "padding_separation", axes[0], "padding separation (hatched: rejected)",
                         "Oranges", ".3f", marked_windows)
plot_window_grid_heatmap(grid_summary, "energy_score", axes[1],
                         f"energy score ({search.energy_criterion}) on {label_names[search.energy_reference_label]}",
                         "Blues", ".3f", marked_windows)
save_figure(figure, preprocessing.figures_dir / f"window_search_grid_{config.waveforms.source}.png")
plt.show()

figure, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
plot_window_tradeoff(window_summary, window_summary["pareto"], search.max_padding_separation, marked_windows, ax)
save_figure(figure, preprocessing.figures_dir / f"window_search_tradeoff_{config.waveforms.source}.png")
plt.show()

## 3. Visual check of the cut

The full stored trace in light grey and the analysis window in dark grey (shaded):
random records of each class, then the records that keep the smallest share of their energy.

In [ ]:
N_RANDOM_EXAMPLES = 2
N_LOWEST_ENERGY_EXAMPLES = 2

random_examples = windows.groupby(columns.label, group_keys=False).sample(
    n=N_RANDOM_EXAMPLES, random_state=config.waveforms.random_seed
)
lowest_energy = windows.nsmallest(N_LOWEST_ENERGY_EXAMPLES, "energy_fraction_kept")
examples = pd.concat([random_examples, lowest_energy])
example_traces = load_waveforms(records.loc[examples.index], config)

for (index, window), trace in zip(examples.iterrows(), example_traces):
    title = (f"{label_names[window[columns.label]]}: {window['trace_name_old']} - "
             f"energy kept {window['energy_fraction_kept']:.1%}")
    figure = plot_window_overlay(trace, sampling_rate_hz, int(window["window_start"]), int(window["window_stop"]),
                                 CV_COMPONENTS, title)
    save_figure(figure, preprocessing.figures_dir / f"window_{window['trace_name_old']}.png")
    plt.show()

## 4. Spectrogram and scalogram of one record

Comparison figure for a record of the station with most records in the inspected set
(waveform of the window, spectrogram and scalogram; one row per component).

In [ ]:
example_station = records[columns.station_id].value_counts().index[0]
station_record = records[records[columns.station_id] == example_station].iloc[[0]]
station_trace = load_waveforms(station_record, config)[0]
processed = process_record(station_trace, sampling_rate_hz, preprocessing, grid, tolerance)
station_db = {name: power_to_db(processed.power[name], preprocessing.reference_power, preprocessing.floor_power)
              for name in REPRESENTATIONS}

record_name = station_record["trace_name_old"].iloc[0]
figure = plot_representation_comparison(
    processed.window_waveform, preprocessing.target_sampling_rate_hz,
    station_db["spectrogram"], station_db["scalogram"],
    grid.frequency_edges_hz, grid.times_s, CV_COMPONENTS,
    f"{label_names[station_record[columns.label].iloc[0]]}: {record_name}",
)
save_figure(figure, preprocessing.figures_dir / f"comparison_{example_station}.png")
print({name: values.shape for name, values in station_db.items()})

## 5. Power distributions and decibel floor

Power values of a sample of records, pooled over components, frequencies and times.
Zeros (components without data, windows reaching beyond the recording) cannot be converted to decibels:
they are raised to `decibel.floor_power`. The floor should lie below the weakest real values (instrument noise),
so that it marks missing data without clipping real signal. Update `floor_power` in the configuration if needed.

In [ ]:
distribution_records = sample_records_per_label(
    records,
    label_column=columns.label,
    group_column=columns.event_id,
    records_per_label=preprocessing.n_distribution_records // 2,
    max_records_per_group=None,
    seed=config.waveforms.random_seed,
)
pooled = pooled_power_values(
    iter_waveform_batches(distribution_records, config), sampling_rate_hz, preprocessing, tolerance
)
QUANTILES = [0.0, 0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 1.0]
summary = pd.DataFrame({name: np.quantile(values, QUANTILES) for name, values in pooled.items()}, index=QUANTILES)
summary.index.name = "quantile"
for name, values in pooled.items():
    print(f"{name}: zeros {np.mean(values == 0):.4%}, below the current floor "
          f"({preprocessing.floor_power:g}) {np.mean(values < preprocessing.floor_power):.4%}")
summary

In [ ]:
figure, ax = plt.subplots(figsize=(10, 4))
positive_db = {name: 10.0 * np.log10(values[values > 0] / preprocessing.reference_power) for name, values in pooled.items()}
plot_value_distributions(positive_db, ax, "power [dB re reference_power] (zeros excluded)", quantile_range=(0.001, 1.0))
ax.axvline(10.0 * np.log10(preprocessing.floor_power / preprocessing.reference_power), color="black", linewidth=1.0,
           linestyle="--")
ax.set_title("dashed line: current floor_power; x range from the 0.1st percentile", loc="left")
figure.tight_layout()
save_figure(figure, preprocessing.figures_dir / f"power_distributions_{config.waveforms.source}.png")

## 6. Compute and store all records

One HDF5 file per waveform source in `preprocessing.output_dir`, with the decibel arrays of both representations
(float16, not yet standardized: the standardization statistics will be computed on the training set only),
the grid and the preprocessing parameters; `records.csv` next to it links each row (`feature_index`) to its record.

In [ ]:
features_path = feature_file_path(preprocessing, config.waveforms.source)
stored_records = preprocess_and_store(
    records, iter_waveform_batches(records, config), sampling_rate_hz, preprocessing, tolerance, features_path
)
print(f"Stored {len(stored_records):,} records in {features_path}")

## 7. PNG images for visual inspection

Comparison figures read back from the stored arrays (spectrogram and scalogram) for a few records of each class,
saved in `figures_dir/png_checks/`. These images are for inspection only; the networks use the stored arrays.

In [ ]:
png_records = stored_records.groupby(columns.label, group_keys=False).sample(
    n=preprocessing.n_png_records_per_class, random_state=config.waveforms.random_seed
)
rows = png_records[FEATURE_INDEX_COLUMN].to_numpy()
stored_db = {name: read_features(features_path, name, rows)[0] for name in REPRESENTATIONS}
png_traces = load_waveforms(records.loc[png_records.index], config)

for position, ((index, record), trace) in enumerate(zip(png_records.iterrows(), png_traces)):
    window_waveform = process_record(trace, sampling_rate_hz, preprocessing, grid, tolerance).window_waveform
    figure = plot_representation_comparison(
        window_waveform, preprocessing.target_sampling_rate_hz,
        stored_db["spectrogram"][position], stored_db["scalogram"][position],
        grid.frequency_edges_hz, grid.times_s, CV_COMPONENTS,
        f"{label_names[record[columns.label]]}: {record['trace_name_old']}",
    )
    save_figure(figure, preprocessing.figures_dir / "png_checks" / f"{label_names[record[columns.label]]}_{record['trace_name_old']}.png")
    plt.close(figure)
print(f"Saved {len(png_records)} PNG images in {preprocessing.figures_dir / 'png_checks'}")